# 🚗 Instructor Demo: Car Weight and Fuel Economy
*Elements of Data Science — Regression Microprojects* <br>
A fully worked demonstration to project in class **before** students start their microprojects. It walks through the same parts as the student notebooks, using a data set none of them have, and adds a set of "tricks": habits and checks that separate a good regression analysis from a mechanical one.

> **Research question:** How much does a car's weight affect its fuel economy — and is a straight line the right model?

|Trick|Idea|Where students use it|
|---|---|---|
|1|Look first: same *r*, very different data (Anscombe)|Every project, Part 3|
|2|Guess *r* before computing; read the sign from the quadrants|Part 3|
|3|Choose units so the slope reads well; *r* has no units|Part 4|
|4|Regression is not symmetric: swapping $x$ and $y$ changes the line|Part 4|
|5|Regression to the mean in standard units|MP2|
|6|Read the error plot; transform to straighten a curve|MP1, MP5|
|7|Split by group: one line or several?|MP2, MP3, MP4|
|8|One point can move the line|MP3|
|9|`np.polyfit` as a check; a curve is not always better|Part 5|
|10|From a working plot to a publication figure; submit|Part 7|

**Review:** [Ch. 15.1 Correlation](https://inferentialthinking.com/chapters/15/1/Correlation.html), [Ch. 15.2 The Regression Line](https://inferentialthinking.com/chapters/15/2/Regression_Line.html), [Ch. 15.3 Least Squares](https://inferentialthinking.com/chapters/15/3/Method_of_Least_Squares.html), [Ch. 15.5 Visual Diagnostics](https://inferentialthinking.com/chapters/15/5/Visual_Diagnostics.html)

## ⚙️ Initialization

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "shared-public")))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "shared-public")))   # Fall 2026/shared-public
sys.path.insert(0, "/home/jovyan/shared-public")
from init import *
from microproject_tools import submit_plot

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

## 1. The Regression Toolbox (from Lab 8)
These are exactly the four functions students wrote in Lab 8, Questions 1.5 and 4.4, and paste into Part 1 of their microproject.

In [ ]:
def standard_units(any_numbers):
    "Convert an array of numbers to standard units."
    return (any_numbers - np.mean(any_numbers)) / np.std(any_numbers)

def correlation(t, label_x, label_y):
    "The correlation coefficient r between two columns of Table t."
    return np.mean(standard_units(t.column(label_x)) * standard_units(t.column(label_y)))

def slope(t, label_x, label_y):
    "Slope of the regression line, in original units."
    r = correlation(t, label_x, label_y)
    return r * np.std(t.column(label_y)) / np.std(t.column(label_x))

def intercept(t, label_x, label_y):
    "Intercept of the regression line, in original units."
    return np.mean(t.column(label_y)) - slope(t, label_x, label_y) * np.mean(t.column(label_x))

## 2. Look First — Anscombe's Quartet
Statistician Francis Anscombe built four small data sets in 1973 to make one point: **numbers alone can hide everything.**

> ### 🎓 Trick 1: Show the numbers, then the pictures
> Compute the line for all four data sets *before* plotting. Students see four identical answers and assume the data are similar. Then reveal the plots.

In [ ]:
anscombe = Table.read_table('anscombe.csv')
summary = Table(make_array('dataset', 'r', 'slope', 'intercept', 'mean of y'))
for d in make_array('I', 'II', 'III', 'IV'):
    t = anscombe.where('dataset', d)
    summary = summary.with_row(make_array(d, round(correlation(t, 'x', 'y'), 3), round(slope(t, 'x', 'y'), 3),
                                          round(intercept(t, 'x', 'y'), 2), round(np.mean(t.column('y')), 2)))
summary

<font color=green>**🙋 Ask the class:** All four data sets have the same *r*, slope, intercept, and average. Sketch what you think each scatter plot looks like.</font>

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), sharey=True)
line_x = make_array(3, 20)
for ax, d in zip(axes, make_array('I', 'II', 'III', 'IV')):
    t = anscombe.where('dataset', d)
    ax.scatter(t.column('x'), t.column('y'), s=40)
    ax.plot(line_x, slope(t, 'x', 'y') * line_x + intercept(t, 'x', 'y'), color='red')
    ax.set_title(f'Data set {d}:  r = {correlation(t, "x", "y"):.2f}')
plt.tight_layout(); plt.show()

Only data set I suits a straight line. II is a **curve**, III has one **outlier** pulling the line, and IV's correlation comes almost entirely from **one point**. This is why every microproject starts with a scatter plot and ends with an error plot.

## 3. Meet the Data
The *Auto MPG* data set describes 392 car models sold in the US from 1970 to 1982, a period of rising gasoline prices.
- **name**, **origin** (usa, europe, japan), **model year**, **cylinders**, **horsepower**
- **weight (lb)**: in pounds
- **mpg**: fuel economy, in miles per gallon (higher is better)

*Source:* Quinlan (1993), [UCI Machine Learning Repository: Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg), via [seaborn-data](https://github.com/mwaskom/seaborn-data). Rows with missing horsepower removed.

In [ ]:
cars = Table.read_table('auto_mpg.csv')
cars

In [ ]:
# Quick exploration: a few questions answered with Table methods
print(cars.num_rows, 'cars')
cars.select('origin', 'weight (lb)', 'mpg').group('origin', np.mean)

## 4. Look First, Then Guess

In [ ]:
cars.scatter('weight (lb)', 'mpg')
plt.show()

> ### 🎓 Trick 2: Guess r, then read its sign from the quadrants
> Have students write down a guess for *r* (most guess too close to −1 or too close to 0). Then show where the products of standard units are positive and negative.

<font color=green>**🙋 Ask the class:** Direction? Form? Strength? Guess *r* to one decimal place.</font>

In [ ]:
xs = standard_units(cars.column('weight (lb)'))
ys = standard_units(cars.column('mpg'))
products = xs * ys
plt.figure(figsize=(6, 6))
plt.scatter(xs, ys, c=np.where(products > 0, 'tab:blue', 'tab:orange'), s=12)
plt.axhline(0, color='black', lw=1); plt.axvline(0, color='black', lw=1)
plt.xlabel('weight (standard units)'); plt.ylabel('mpg (standard units)')
plt.title('Blue: product > 0      Orange: product < 0')
plt.show()
print('Points with a negative product:', np.count_nonzero(products < 0), 'of', len(products))
print('r = average of the products =', round(np.mean(products), 3))

Most points sit in the upper-left and lower-right quadrants, so the average product, *r*, is **negative**. This is the first negative correlation students meet in the microprojects.

## 5. Fit the Line and Interpret It

In [ ]:
r_cars = correlation(cars, 'weight (lb)', 'mpg')
slope_lb = slope(cars, 'weight (lb)', 'mpg')
intercept_lb = intercept(cars, 'weight (lb)', 'mpg')
print(f'r = {r_cars:.3f}')
print(f'mpg = {slope_lb:.5f} × weight (lb) + {intercept_lb:.2f}')

> ### 🎓 Trick 3: Pick units that make the slope readable — r does not care
> A slope of −0.0076 mpg per pound is correct but hard to say aloud. Rescale weight to thousands of pounds: the slope changes by a factor of 1000 but *r* stays exactly the same, because standard units have no units.

In [ ]:
cars = cars.with_columns('weight (1000 lb)', cars.column('weight (lb)') / 1000)
slope_k = slope(cars, 'weight (1000 lb)', 'mpg')
print(f'r with pounds:          {correlation(cars, "weight (lb)", "mpg"):.4f}')
print(f'r with thousands of lb: {correlation(cars, "weight (1000 lb)", "mpg"):.4f}')
print(f'Each extra 1,000 lb of car is predicted to cost {abs(slope_k):.1f} mpg.')

**Prediction:** a 3,000 lb car.

In [ ]:
mpg_3000 = slope_lb * 3000 + intercept_lb
mpg_3000

> ### 🎓 Trick 4: Swap x and y on purpose
> A classic student bug is reversing the columns. The reversed line is *not* the same line solved the other way: its slope is not 1/slope. Both lines are 'slope = r' in standard units, so they only agree when |r| = 1.

In [ ]:
weight_on_mpg = slope(cars, 'mpg', 'weight (lb)')
print(f'1 / (mpg-on-weight slope) = {1 / slope_lb:,.0f} lb per mpg')
print(f'weight-on-mpg slope       = {weight_on_mpg:,.0f} lb per mpg')

# Draw both lines on the weight–mpg plot
line_w = make_array(1600, 5200)
plt.figure(figsize=(8, 5))
plt.scatter(cars.column('weight (lb)'), cars.column('mpg'), s=10, alpha=0.4)
plt.plot(line_w, slope_lb * line_w + intercept_lb, color='red', lw=2, label='predict mpg from weight')
mpg_line = (line_w - intercept(cars, 'mpg', 'weight (lb)')) / weight_on_mpg     # same line, solved for mpg
plt.plot(line_w, mpg_line, color='purple', lw=2, ls='--', label='predict weight from mpg')
plt.xlabel('weight (lb)'); plt.ylabel('mpg'); plt.legend(); plt.show()

<font color=green>**🙋 Ask the class:** Both lines pass through one point. Which one, and why?  *(The point of averages.)*</font>

> ### 🎓 Trick 5: Regression to the mean, with numbers
> In standard units the prediction is r × x. A car 2 SDs heavier than average is predicted to be only about 1.7 SDs below average in mpg, not 2.

In [ ]:
z_weight = 2
z_mpg_pred = r_cars * z_weight
print(f'2 SD above average weight  →  predicted {z_mpg_pred:.2f} SD in mpg')
print(f'In original units: {np.mean(cars.column("weight (lb)")) + 2 * np.std(cars.column("weight (lb)")):,.0f} lb'
      f'  →  {np.mean(cars.column("mpg")) + z_mpg_pred * np.std(cars.column("mpg")):.1f} mpg')

## 6. Check the Fit

> ### 🎓 Trick 6: Read the error plot, then transform
> A large |r| does not mean a straight line is right. Plot the errors: a curve in the errors means a curve in the data. Here the physics suggests a better variable: the fuel a car *uses* (gallons per 100 miles) should grow roughly in proportion to its mass, since moving more mass takes more energy.

In [ ]:
weight = cars.column('weight (lb)')
mpg = cars.column('mpg')
errors = mpg - (slope_lb * weight + intercept_lb)

# Graph of averages: average mpg in 250-lb bins of weight
binned = cars.with_columns('weight bin', np.round(weight / 250) * 250)
averages = binned.select('weight bin', 'mpg').group('weight bin', np.mean)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].scatter(weight, mpg, s=10, alpha=0.35)
axes[0].scatter(averages.column('weight bin'), averages.column('mpg mean'), marker='D', color='red', s=40, label='average in bin')
axes[0].plot(line_w, slope_lb * line_w + intercept_lb, color='black')
axes[0].set_xlabel('weight (lb)'); axes[0].set_ylabel('mpg'); axes[0].legend(); axes[0].set_title('Averages bend away from the line')
axes[1].scatter(weight, errors, s=10, alpha=0.5)
axes[1].axhline(0, color='red')
axes[1].set_xlabel('weight (lb)'); axes[1].set_ylabel('error (mpg)'); axes[1].set_title('Errors: a U shape, not a cloud')
plt.show()

In [ ]:
cars = cars.with_columns('gallons per 100 miles', 100 / cars.column('mpg'))
gp = cars.column('gallons per 100 miles')
r_gp = correlation(cars, 'weight (lb)', 'gallons per 100 miles')
slope_gp = slope(cars, 'weight (lb)', 'gallons per 100 miles')
intercept_gp = intercept(cars, 'weight (lb)', 'gallons per 100 miles')
errors_gp = gp - (slope_gp * weight + intercept_gp)
print(f'mpg:                   r = {r_cars:.3f}')
print(f'gallons per 100 miles: r = {r_gp:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].scatter(weight, gp, s=10, alpha=0.4)
axes[0].plot(line_w, slope_gp * line_w + intercept_gp, color='black')
axes[0].set_xlabel('weight (lb)'); axes[0].set_ylabel('gallons per 100 miles'); axes[0].set_title(f'Transformed: r = {r_gp:.2f}')
axes[1].scatter(weight, errors_gp, s=10, alpha=0.5)
axes[1].axhline(0, color='red')
axes[1].set_xlabel('weight (lb)'); axes[1].set_ylabel('error (gal / 100 mi)'); axes[1].set_title('Errors: much closer to a patternless cloud')
plt.show()

Predictions still come back to the units people care about: predict gallons per 100 miles, then convert.

In [ ]:
gp_3000 = slope_gp * 3000 + intercept_gp
print(f'3,000 lb car:  {gp_3000:.2f} gal/100 mi  =  {100 / gp_3000:.1f} mpg   (straight-line mpg model said {mpg_3000:.1f})')

<font color=green>**🙋 Ask the class:** Notice the errors still spread out more for heavy cars (a 'fan'). Is that a problem for prediction? For which cars are we least certain?</font>

## 7. One Line or Several?

> ### 🎓 Trick 7: Split by a group variable
> Color the points by a third variable before trusting one line. Here, US cars are much heavier on average; part of the overall trend is the difference *between* countries. The same idea drives the twists in MP2 (gender), MP3 (smoking), and MP4 (species).

In [ ]:
origins = make_array('usa', 'europe', 'japan')
colors = make_array('tab:blue', 'tab:orange', 'tab:green')
group_fits = Table(make_array('origin', 'cars', 'mean weight', 'r', 'slope per 1000 lb'))
plt.figure(figsize=(9, 5.5))
for i in np.arange(3):
    t = cars.where('origin', origins.item(i))
    s = slope(t, 'weight (lb)', 'gallons per 100 miles'); b = intercept(t, 'weight (lb)', 'gallons per 100 miles')
    xs = make_array(min(t.column('weight (lb)')), max(t.column('weight (lb)')))
    plt.scatter(t.column('weight (lb)'), t.column('gallons per 100 miles'), s=12, alpha=0.5, color=colors.item(i))
    plt.plot(xs, s * xs + b, color=colors.item(i), lw=3, label=origins.item(i))
    group_fits = group_fits.with_row(make_array(origins.item(i), t.num_rows, round(np.mean(t.column('weight (lb)'))),
                                                round(correlation(t, 'weight (lb)', 'gallons per 100 miles'), 2), round(s * 1000, 2)))
plt.xlabel('weight (lb)'); plt.ylabel('gallons per 100 miles'); plt.legend(); plt.show()
group_fits

<font color=green>**🙋 Ask the class:** Within each country *r* is lower than for all cars together. Why? And are the slopes similar? What would similar slopes mean physically?</font>

## 8. One Point Can Move the Line

> ### 🎓 Trick 8: Add an imaginary car
> Points far from the average $x$ have the most pull (Anscombe IV). Add one made-up heavy car with excellent mileage and watch the line and *r* change. Then add the same car at the average weight: the slope does not move at all.

In [ ]:
def fit_with_extra(extra_weight, extra_gp):
    "r and slope (per 1000 lb) after adding one imaginary car."
    t = cars.select('weight (lb)', 'gallons per 100 miles').with_row(make_array(extra_weight, extra_gp))
    return correlation(t, 'weight (lb)', 'gallons per 100 miles'), slope(t, 'weight (lb)', 'gallons per 100 miles') * 1000

average_weight = np.mean(weight)
for label, w in [('real data only', None), ('+ 6,000 lb car at 2 gal/100 mi', 6000),
                 (f'+ {average_weight:,.0f} lb (average) car at 2', average_weight)]:
    r_new, s_new = (r_gp, slope_gp * 1000) if w is None else fit_with_extra(w, 2)
    print(f'{label:36s}  r = {r_new:.3f}   slope = {s_new:.3f} gal/100 mi per 1000 lb')

## 9. Check With `np.polyfit` — and a Warning About Curves

> ### 🎓 Trick 9: polyfit agrees with the toolbox; a curve is not automatically better
> For degree 1, `np.polyfit` must reproduce the toolbox line. Degree 2 lowers the sum of squared errors on mpg, but a lower SSE on the *same* data is guaranteed for any more flexible model. A physically motivated transform (Trick 6) is easier to explain than a parabola.

In [ ]:
coefficients = np.polyfit(weight, mpg, 1)
print('polyfit deg 1:', coefficients, '   toolbox:', slope_lb, intercept_lb)
print('agree?', np.isclose(coefficients.item(0), slope_lb) and np.isclose(coefficients.item(1), intercept_lb))

quad = np.polyfit(weight, mpg, 2)
sse_line = np.sum((mpg - (slope_lb * weight + intercept_lb))**2)
sse_quad = np.sum((mpg - (quad.item(0) * weight**2 + quad.item(1) * weight + quad.item(2)))**2)
print(f'SSE straight line: {sse_line:,.0f}     SSE parabola: {sse_quad:,.0f}')

**Common bugs to watch for while circulating**

|Symptom|Likely bug|
|---|---|
|Toolbox test gives r ≠ 1|`standard_units` uses the wrong array, or divides by the mean|
|r far outside −1 to 1|Products added with `np.sum` instead of averaged with `np.mean`|
|Line misses the point of averages|Intercept uses $\bar{x} - \text{slope}\cdot\bar{y}$ (swapped)|
|Line looks far too steep or flat|Arguments reversed: `slope(t, y, x)` (Trick 4)|
|Students in one project disagree on r|One used a subset (the twist) for the final numbers|
|`submit_plot` saves a blank image|Plot drawn before `fig = plt.figure(...)` or with `Table.scatter`, which makes its own figure|

## 10. The Final Figure

> ### 🎓 Trick 10: Show a bad figure next to a good one
> Ask students to list everything wrong with the left figure. Their list becomes the requirements checklist in Part 7 of their notebooks.

In [ ]:
fig_bad = plt.figure(figsize=(6, 4))
plt.scatter(weight, gp)
plt.plot(line_w, slope_gp * line_w + intercept_gp)
plt.show()

<font color=green>**🙋 Ask the class:** What's missing? *(Axis labels and units, a title with a finding, a legend with the equation and r, groups, transparency for overlapping points…)*</font>

In [ ]:
fig = plt.figure(figsize=(9, 5.5))
for i in np.arange(3):
    t = cars.where('origin', origins.item(i))
    plt.scatter(t.column('weight (lb)'), t.column('gallons per 100 miles'), s=14, alpha=0.45, color=colors.item(i), label=f'{origins.item(i)} cars')
plt.plot(line_w, slope_gp * line_w + intercept_gp, color='black', lw=2.5,
         label=f'all cars: gal/100 mi = {slope_gp * 1000:.2f} × (weight / 1000 lb) {intercept_gp:+.2f},  r = {r_gp:.2f}')
plt.xlabel('car weight (lb)'); plt.ylabel('fuel use (gallons per 100 miles)')
plt.title(f'Every extra 1,000 lb costs about {slope_gp * 1000:.1f} gallons per 100 miles')
plt.legend(); plt.show()

Submitting works exactly as it does for students. The `'DEMO'` id keeps it separate from the five projects in the gallery.

In [ ]:
instructor_name = 'Instructor Demo'
caption = 'Fuel use rises in a straight line with weight once mpg is converted to gallons per 100 miles (r = 0.89).'
submit_plot(fig, instructor_name, 'DEMO', r=r_gp, slope=slope_gp, intercept=intercept_gp, caption=caption)

## Bridge to the Microprojects
|Project|Which tricks matter most|
|---|---|
|MP1 🌍 CO₂|6 (errors curve, r = 0.99 anyway), 9 (extrapolation to 2050)|
|MP2 📏 Galton|5 (regression to the mean), 7 (sons vs daughters)|
|MP3 👶 Births|7 (smokers vs non-smokers), 8 (extreme gestation lengths)|
|MP4 🐧 Penguins|7 (species), 2 (guessing r with clusters)|
|MP5 ⚛️ Elements|6 (ranking errors), 7 (one period), and correlation vs causation|

**Closing question:** Weight and fuel use are strongly associated, and here physics gives a causal mechanism. In which microproject is a causal claim hardest to defend, and why?